Migrated from Databricks to Neuron (auto-port).

In [ ]:
import os
import sys
from pathlib import Path

for _candidate in (
    "/home/jovyan/work/default/rashmi-ranjan/lib",
    str(Path.cwd().parent / "lib"),
    str(Path.cwd() / "lib"),
):
    if _candidate not in sys.path and Path(_candidate).is_dir():
        sys.path.insert(0, _candidate)

from neuron_runtime import (
    GoogleSheetReader,
    SlackReportsTrigger,
    get_sql_df,
    load_service_account,
    load_slack_token,
    make_unique_columns,
)

service_account = load_service_account()
slack_token = load_slack_token()

In [ ]:
# MAGIC
# MAGIC import trino
# MAGIC import pandas as pd
# MAGIC
# MAGIC conn = trino.dbapi.connect(
# MAGIC     host="trino-tableau-prod.dp.navi-tech.in",
# MAGIC     port=443,
# MAGIC     user="databricks-user",
# MAGIC     catalog="awsdatacatalog",
# MAGIC     schema="default",
# MAGIC     http_scheme="https",
# MAGIC )
# MAGIC
# MAGIC
# MAGIC query = """
# MAGIC -- MTD Tickets Handled by Agent (SPADE Template Review V1)
# MAGIC -- Returns count of distinct tickets per agent for current month to date
# MAGIC WITH base_table AS (
# MAGIC     SELECT 
# MAGIC         a."Annotation Date",
# MAGIC         a."Ticket Id",
# MAGIC         a."Project Type",
# MAGIC         a."Email ID",
# MAGIC         b.td AS "Time Taken (Seconds)"
# MAGIC     FROM (
# MAGIC         SELECT 
# MAGIC             annotated_at AS "Annotation Date",
# MAGIC             ticket_id AS "Ticket Id",
# MAGIC             project_type AS "Project Type",
# MAGIC             p1.email_id AS "Email ID"
# MAGIC         FROM corp_analytics.data_annotation_projects_p1_tbl p1
# MAGIC         WHERE project_type NOT IN ('App Tagging', 'Address Tagging') 
# MAGIC         
# MAGIC         UNION ALL
# MAGIC         SELECT annotated_at, ticket_id, 'Address annotation metaaddress', ma.email_id
# MAGIC         FROM corp_analytics_databricks.data_annotation_meta_address ma
# MAGIC         
# MAGIC         UNION ALL
# MAGIC         SELECT annotated_at, ticket_id, 'Address annotation clustering', ma.email_id
# MAGIC         FROM corp_analytics.data_annotation_address_clustering_tbl ma
# MAGIC         
# MAGIC         UNION ALL
# MAGIC         SELECT annotated_at, ticket_id, 'speech-to-text-test-whisper', ma.email_id
# MAGIC         FROM corp_analytics.data_annotation_speech_to_text_tbl ma
# MAGIC         
# MAGIC         UNION ALL
# MAGIC         SELECT annotated_at, ticket_id, 'Suspicious feedback tagging', ma.email_id
# MAGIC         FROM corp_analytics_databricks.data_annotation_Suspicious_feedback_tagging_v2_tbl ma
# MAGIC         
# MAGIC         UNION ALL
# MAGIC         SELECT CAST("Annotation Date" AS TIMESTAMP), "Ticket Id", "Project Type", email_id
# MAGIC         FROM corp_analytics_databricks.data_annotation_spade_template_tagging_v3_tbl
# MAGIC         
# MAGIC         UNION ALL
# MAGIC         SELECT CAST("Annotation Date" AS TIMESTAMP), "Ticket Id", "Project Type", email_id
# MAGIC         FROM corp_analytics_databricks.data_annotation_spade_template_113_tagging_v3_tbl
# MAGIC         
# MAGIC         UNION ALL
# MAGIC         SELECT CAST("Annotation Date" AS TIMESTAMP), "Ticket Id", "Project Type", email_id
# MAGIC         FROM corp_analytics_databricks.data_annotation_spade_adf_tagging_v3_tbl
# MAGIC         
# MAGIC         UNION ALL
# MAGIC         SELECT annotated_at, ticket_id, 'AISHA HRC DA Tagging', email_id
# MAGIC         FROM corp_analytics_databricks.data_annotation_aisha_hrc_chatbot_tbl ma
# MAGIC         
# MAGIC         UNION ALL
# MAGIC         SELECT annotated_at, ticket_id, 'GenAI aisha chat QC DA Team V1', email_id
# MAGIC         FROM corp_analytics_databricks.GenAI_aisha_chat_QC_DA_Team_V1_tbl ma
# MAGIC         
# MAGIC         UNION ALL
# MAGIC         SELECT annotated_at, ticket_id, 'Address Entity Tagging', email_id
# MAGIC         FROM corp_analytics_databricks.data_annotation_address_entity_tagging_tbl ma
# MAGIC         
# MAGIC         UNION ALL
# MAGIC         SELECT annotated_at, ticket_id, 'SPADE Category Matching V1', email_id
# MAGIC         FROM corp_analytics_databricks.data_annotation_spade_category_V2_tbl ma
# MAGIC         
# MAGIC         UNION ALL
# MAGIC         SELECT annotated_at, ticket_id, 'SPADE Category Matching V1', email_id
# MAGIC         FROM corp_analytics_databricks.data_annotation_spade_category_V3_tbl ma
# MAGIC         
# MAGIC         UNION ALL
# MAGIC         SELECT annotated_at, ticket_id, 'SPADE Bank Statement Parser V1', email_id
# MAGIC         FROM corp_analytics_databricks.data_annotation_spade_bank_statement_tbl ma
# MAGIC         
# MAGIC         UNION ALL
# MAGIC         SELECT annotated_at, ticket_id, 'Pincode_qc_v1', email_id
# MAGIC         FROM corp_analytics_databricks.Pincode_qc_v1_tbl ma
# MAGIC         
# MAGIC         UNION ALL
# MAGIC         SELECT CAST(annotated_at AS TIMESTAMP), ticket_id, use_case_type, ma.email_id
# MAGIC         FROM corp_analytics_databricks.call_annotation_v3_tbl ma
# MAGIC     ) a
# MAGIC     LEFT JOIN (
# MAGIC         SELECT ticket_id, td 
# MAGIC         FROM corp_analytics_databricks.data_annotation_online_time_tbl
# MAGIC     ) b ON a."Ticket Id" = b.ticket_id
# MAGIC     WHERE 
# MAGIC         b.td IS NOT NULL 
# MAGIC         AND a."Annotation Date" IS NOT NULL 
# MAGIC         AND a."Ticket Id" IS NOT NULL 
# MAGIC         AND a."Email ID" IS NOT NULL
# MAGIC )
# MAGIC SELECT 
# MAGIC     TRIM(COALESCE(u.name, bt."Email ID")) AS "Agent Name",
# MAGIC     bt."Project Type",
# MAGIC     COUNT(DISTINCT bt."Ticket Id") AS "MTD Tickets Handled"
# MAGIC FROM base_table bt
# MAGIC LEFT JOIN mjolnir_service.users u ON u.email_id = bt."Email ID"
# MAGIC WHERE 
# MAGIC     DATE_TRUNC('month', CAST(bt."Annotation Date" AS DATE)) = DATE_TRUNC('month', CURRENT_DATE)
# MAGIC     AND CAST(bt."Annotation Date" AS DATE) <= CURRENT_DATE
# MAGIC     AND bt."Project Type" = 'SPADE Template Review V1'
# MAGIC GROUP BY 
# MAGIC     TRIM(COALESCE(u.name, bt."Email ID")),
# MAGIC     bt."Project Type"
# MAGIC ORDER BY 
# MAGIC     "Agent Name",
# MAGIC     "Project Type"
# MAGIC
# MAGIC """
# MAGIC
# MAGIC df = pd.read_sql(query, conn)
# MAGIC display(df)

In [ ]:
# Python cell
import os
import shutil

if not os.path.exists("data"):
    os.makedirs("data")

gsheet_url = "https://docs.google.com/spreadsheets/d/1c2SbbJazzI1uRBuymakKD_T-s9uyl580ZCInDgmyBNg/edit?gid=166624487#gid=166624487"
worksheet_name = "Agent MTD Tickets Handled"
gs_reader = GoogleSheetReader(gsheet_url, service_account)

gs_reader.clear_sheets(
    worksheet_name,
    start_row=1,
    start_column=1
)

combined_df = pd.concat([df], ignore_index=True) 

gs_reader.set_df_in_sheets( 
worksheet_name, 
combined_df, 
include_headers=True, 
row=1, 
col=1
)